In [1]:
import xarray as xr
import numpy as np
import matplotlib.pylab as plt
import matplotlib.cm as cm
import cmocean.cm as cmo
from scipy.interpolate import CloughTocher2DInterpolator, LinearNDInterpolator, NearestNDInterpolator
import gribscan
import intake
import uxarray as ux
from scipy.interpolate import griddata
# from tqdm import tqdm

In [2]:
nz1  = [2.5, 7.5, 12.5, 17.5, 22.5, 27.5, 32.5, 37.5, 42.5, 47.5, 52.5, 57.5,
    62.5, 67.5, 72.5, 77.5, 82.5, 87.5, 92.5, 97.5, 105, 115, 125, 135, 145,
    155, 165, 175, 185, 195, 210, 230, 250, 270, 290, 320, 360, 400, 440,
    480, 520, 560, 600, 640, 710, 810, 950, 1110, 1255, 1415, 1600, 1810,
    2035, 2275, 2525, 2775, 3025, 3275, 3525, 3775, 4025, 4275, 4525, 4775,
    5025, 5275, 5525, 5825, 6175, 6175]

nz = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85,
    90, 95, 100, 110, 120, 130, 140, 150, 160, 170, 180, 190, 200, 220, 240,
    260, 280, 300, 340, 380, 420, 460, 500, 540, 580, 620, 660, 760, 860,
    1040, 1180, 1330, 1500, 1700, 1920, 2150, 2400, 2650, 2900, 3150, 3400,
    3650, 3900, 4150, 4400, 4650, 4900, 5150, 5400, 5650, 6000, 6350]

In [3]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="zarr.core.metadata.v2")
cat = intake.open_catalog("https://data.nextgems-h2020.eu/catalog.yaml")

In [6]:
ds = cat.FESOM["IFS_4.4-FESOM_5-cycle3"]["3D_daily_native"].reader.read(
    engine="netcdf4",
    chunks={
    "time": 30,     # 1 month per chunk
    "lev": 10,      # vertical levels (e.g., 10 levels per chunk)
    "value": 1_000_000,  # number of spatial points per chunk
    },
)
ds = ds[['u', 'v','w', 'salt']]
ds

<xarray.Dataset> Size: 22TB
Dimensions:  (time: 1808, nz1: 69, elem: 14741520, nz: 70, nod2: 7402886)
Coordinates:
    lon      (nod2) float64 59MB dask.array<chunksize=(7402886,), meta=np.ndarray>
    lat      (nod2) float64 59MB dask.array<chunksize=(7402886,), meta=np.ndarray>
  * nz       (nz) float64 560B 0.0 5.0 10.0 15.0 ... 5.65e+03 6e+03 6.35e+03
  * time     (time) datetime64[ns] 14kB 2020-01-20T23:56:00 ... 2024-12-31T23...
  * nz1      (nz1) float64 552B 2.5 7.5 12.5 ... 5.525e+03 5.825e+03 6.175e+03
Dimensions without coordinates: elem, nod2
Data variables:
    u        (time, nz1, elem) float32 7TB dask.array<chunksize=(12, 1, 460673), meta=np.ndarray>
    v        (time, nz1, elem) float32 7TB dask.array<chunksize=(12, 1, 460673), meta=np.ndarray>
    w        (time, nz, nod2) float32 4TB dask.array<chunksize=(12, 2, 321865), meta=np.ndarray>
    salt     (time, nz1, nod2) float32 4TB dask.array<chunksize=(12, 2, 321865), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.4
    history:      2022-03-19 08:26:17 GMT; Grid description file generated wi...

In [8]:
ds_subset = ds.isel(time=0,nz1=0,nz=0)
ds_subset

<xarray.Dataset> Size: 296MB
Dimensions:  (elem: 14741520, nod2: 7402886)
Coordinates:
    lon      (nod2) float64 59MB dask.array<chunksize=(7402886,), meta=np.ndarray>
    lat      (nod2) float64 59MB dask.array<chunksize=(7402886,), meta=np.ndarray>
    nz       float64 8B 0.0
    time     datetime64[ns] 8B 2020-01-20T23:56:00
    nz1      float64 8B 2.5
Dimensions without coordinates: elem, nod2
Data variables:
    u        (elem) float32 59MB dask.array<chunksize=(460673,), meta=np.ndarray>
    v        (elem) float32 59MB dask.array<chunksize=(460673,), meta=np.ndarray>
    w        (nod2) float32 30MB dask.array<chunksize=(321865,), meta=np.ndarray>
    salt     (nod2) float32 30MB dask.array<chunksize=(321865,), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.4
    history:      2022-03-19 08:26:17 GMT; Grid description file generated wi...

In [4]:
nodes = cat.FESOM["IFS_4.4-FESOM_5-cycle3"]["node_grid"].reader.read(engine="netcdf4", chunks={})
elems = cat.FESOM["IFS_4.4-FESOM_5-cycle3"]["elem_grid"].reader.read(engine="netcdf4", chunks={})

nodes=nodes.compute()
elems = elems.compute()

In [6]:
print(nodes)
print(elems)

<xarray.Dataset> Size: 3GB
Dimensions:          (grid_size: 7402886, grid_corners: 16, grid_rank: 1,
                      nlinks_max: 8, ntriags: 14741520, Three: 3, nlev: 69,
                      nlev_bnds: 70)
Dimensions without coordinates: grid_size, grid_corners, grid_rank, nlinks_max,
                                ntriags, Three, nlev, nlev_bnds
Data variables: (12/13)
    grid_center_lon  (grid_size) float64 59MB 182.8 182.7 182.7 ... 178.3 178.3
    grid_corner_lon  (grid_size, grid_corners) float64 948MB 182.8 ... 178.3
    grid_center_lat  (grid_size) float64 59MB -78.1 -78.05 ... -77.82 -77.86
    grid_corner_lat  (grid_size, grid_corners) float64 948MB -78.11 ... -77.86
    grid_dims        (grid_rank) float64 8B 7.403e+06
    grid_imask       (grid_size) float64 59MB 1.0 1.0 1.0 1.0 ... 1.0 1.0 1.0
    ...               ...
    node_node_links  (grid_size, nlinks_max) float64 474MB 1.17e+03 ... nan
    triag_nodes      (ntriags, Three) float64 354MB 1.0 1.169e+03 ... 7

In [11]:
import pandas as pd
# Nodes: Write nod2d.out (node_number, lon, lat, flag)
node_count = nodes.sizes["grid_size"]
node_df = pd.DataFrame({
    "node_number": np.arange(1, node_count + 1),
    "lon": nodes["grid_center_lon"].values,
    "lat": nodes["grid_center_lat"].values,
    "flag": nodes["grid_imask"].astype(int).values
})

with open("nod2d.out", "w") as f:
    f.write(f"{node_count}\n")
    node_df.to_csv(f, sep=" ", index=False, header=False, float_format="%.6f")

# Elements: Write elem2d.out (node1, node2, node3)
elem_count = elems.sizes["grid_size"]
elem_df = pd.DataFrame(elems["triag_nodes"].values + 1)  # 1-based indexing

with open("elem2d.out", "w") as f:
    f.write(f"{elem_count}\n")
    elem_df.to_csv(f, sep=" ", index=False, header=False, float_format="%d")

In [12]:
grid_path = "/work/bk1450/b383184/Amazon/NextGEMS/mesh_NG/"

In [ ]:
uxds = ux.open_mfdataset(grid_path, ds_subset)